# Skin Lab (BÀI GIẢI) — Máy tính thay đổi một pixel như thế nào?

Ta sẽ bắt đầu bằng một ảnh rất nhỏ: **7 hàng × 7 cột**. Trong ảnh có vùng da, nền xanh và một chấm đỏ ở giữa.
Ở mỗi bước, em chỉ cần theo dõi pixel giữa. Trang sẽ cho em thấy màu của pixel, các số được lấy ra, phép tính
với những số đó và màu xuất hiện sau phép tính.

Sau khi hiểu sáu bước trên ảnh nhỏ, em sẽ dùng **NumPy** và **SciPy** để làm đúng những việc đó trên cả ảnh.
Cuối bài, **MediaPipe Face Mesh** giới hạn vùng được phép xử lý vào bên trong khuôn mặt.

Đây là bài học về thuật toán xử lý ảnh, **không phải công cụ chẩn đoán hay đánh giá làn da**. Luật màu trong bài
có thể nhận sai khi ánh sáng, camera, màu da hoặc màu nền thay đổi.

Trang tự lưu code, lựa chọn trong các bảng và chặng em đã hoàn thành trên máy này. Em có thể đóng trang rồi quay lại.
Ảnh camera không được lưu. Muốn chuyển bài sang máy khác, em bấm **Tải notebook**.

## Trước khi bắt đầu

Chạy hai ô dưới. Ô đầu mở hình minh họa và camera. Ô thứ hai chuẩn bị NumPy, SciPy, Pillow cùng các con số dùng
trong bài. Em chưa cần sửa hai ô này.

Mỗi nhiệm vụ sau đó đều ghi rõ: dữ liệu nào đã cho sẵn, INPUT nào đến từ bên ngoài, em cần viết gì và OUTPUT nào
chứng minh hàm đã đúng. Sau khi sửa một hàm, chạy ô xem kết quả ngay dưới hàm đó.

In [ ]:
import magic_mirror
magic_mirror.skin_intro()

In [ ]:
import numpy as np
from PIL import Image
from scipy import ndimage


SKIN_VOTE_KERNEL = (
    (1, 1, 1),
    (1, 1, 1),
    (1, 1, 1),
)

SOFTEN_KERNEL = (
    (1, 2, 1),
    (2, 4, 2),
    (1, 2, 1),
)

MASK_OFF, MASK_ON = 0, 255
SKIN_NEIGHBOURS_NEEDED = 5
PIMPLE_RED_GAP = 24

## Ta cần tạo ra những hình nào?

Chạy ô dưới để xem ảnh ban đầu và ba kết quả cần tạo:

1. `skin_mask`: ô trắng là nơi luật màu tạm cho là da; ô đen là nơi không chọn.
2. `pimple_mask`: ô trắng là vùng đỏ nổi bật cần làm mềm; ô đen là nơi giữ nguyên.
3. Ảnh cuối: chỉ những ô trắng trong `pimple_mask` được thay bằng màu đã làm mềm.

Trong hai mask, **255 nghĩa là chọn** và **0 nghĩa là không chọn**. Đây chỉ là hai số giúp chương trình ghi nhớ
vị trí. Chúng không phải màu da và cũng không phải điểm đánh giá da.

In [ ]:
magic_mirror.show_skin_pipeline_overview()

## Cơ chế 1 — Một pixel là ba số RGB

Chạy bảng dưới rồi bấm vào pixel giữa của ảnh 7×7. Pixel đó có màu `(225, 62, 66)`, nghĩa là `R = 225`,
`G = 62`, `B = 66`. Kéo từng thanh để xem khi chỉ một số đổi thì màu của pixel đổi ra sao.

Khi tách kênh R, máy giữ `225` và đặt hai số còn lại về `0`, nên kết quả là `(225, 0, 0)`. Tách G và B cũng
làm đúng một việc như vậy. Bảng sẽ yêu cầu em dùng cơ chế này với một bộ số chưa xuất hiện trong ví dụ.

In [ ]:
magic_mirror.show_mechanism("rgb_pixel")

In [ ]:
magic_mirror.show_skin_pixel_channels()

## Từ một pixel sang cả ảnh bằng NumPy

Trong bảng vừa rồi, em đã chọn một ô bằng hàng và cột. NumPy dùng đúng cách đánh địa chỉ đó:

- `pixels[3, 3]` lấy ba số RGB của pixel ở hàng 3, cột 3.
- `pixels[:, :, 0]` lấy số R của **mọi pixel**.
- `pixels[:, :, 1]` lấy số G của mọi pixel.
- `pixels[:, :, 2]` lấy số B của mọi pixel.

Pillow đọc ảnh. `np.asarray(image)` biến ảnh thành một bảng số NumPy tên là `pixels`. Nếu ảnh cao 60, rộng 80 và có ba
kênh màu thì `pixels.shape` là `(60, 80, 3)`. Chạy hai ô dưới: OUTPUT sẽ ghi kích thước, ba số của một pixel,
rồi hiện ảnh gốc bên cạnh ba kênh R, G, B.

In [ ]:
import numpy as np

sample = magic_mirror.skin_sample_image()
pixels = np.asarray(sample, dtype=np.int16)
print("Kích thước bảng pixels:", pixels.shape, "= chiều cao, chiều rộng, ba kênh RGB")
print("Ba số của pixel da mẫu:", pixels[20, 40])
print("Kích thước bảng kênh đỏ:", pixels[:, :, 0].shape)

In [ ]:
magic_mirror.show_numpy_channels()

## Thư viện làm lại cùng phép tính trên mọi pixel

Các bảng tương tác cho em tính một pixel bằng số nhỏ. Trong dự án, ta gọi các hàm dưới đây để làm phép tính đó
ở mọi vị trí của ảnh:

| Việc cần làm | Hàm thực hiện việc đó |
|---|---|
| Nhân và cộng các số trong vùng 3×3 | `scipy.ndimage.convolve` |
| Tính trung bình các số trong vùng 5×5 | `scipy.ndimage.uniform_filter` |
| Biến một ô được chọn thành vùng 3×3 | `scipy.ndimage.maximum_filter` |
| Chọn màu mới hoặc màu ban đầu cho từng pixel | `np.where` |
| Giữ số màu trong khoảng 0 đến 255 | `np.clip` |
| Biến bảng số trở lại thành ảnh | `Image.fromarray` |

Em không cần tự viết hai vòng `for row` và `for column`. Em cần chọn đúng bảng số đưa vào hàm, rồi kiểm tra
OUTPUT bằng số và hình.

## Từ ba số RGB đến một ô trắng hoặc đen

Ta xét pixel `(183, 127, 103)`. Máy gán `R = 183`, `G = 127`, `B = 103`, rồi thay đúng ba số đó vào:

```text
brightness = (183 + 127 + 103) // 3 = 413 // 3 = 137
warmth = 183 - 103 = 80
red_green_gap = 183 - 127 = 56
```

Ba kết quả đều đạt các điều kiện trong code, nên OUTPUT của pixel này là `255`: ô trắng, được chọn.

Với nền xanh `(35, 80, 185)`, `warmth = 35 - 185 = -150`. Điều kiện cần `warmth >= 8`, nên OUTPUT là `0`:
ô đen, không chọn. Đây là một luật RGB viết tay để học cơ chế; nó không nhận đúng mọi màu da và mọi ánh sáng.

In [ ]:
magic_mirror.show_mechanism("rgb_rule")

### Nhiệm vụ 2 — Hoàn thành `skin_evidence`

- **Dữ liệu đã cho sẵn:** `red`, `green`, `blue` là ba số, hoặc ba bảng số có cùng kích thước.
- **INPUT thật:** chưa có ở đây. Sau này `detect_skin` sẽ đưa màu của ảnh hoặc camera vào hàm.
- **Em cần viết:** tính `warmth = red - blue`, `red_green_gap = red - green`, rồi đưa `looks_like_skin` vào `np.where`.
- **OUTPUT để kiểm tra:** `(183, 127, 103)` trả `255`; `(35, 80, 185)` trả `0`. Nếu INPUT là ba bảng, OUTPUT là
  một bảng `uint8` có cùng số hàng và cột.

In [ ]:
def skin_evidence(red, green, blue):
    """Áp dụng cùng một luật RGB cho một pixel hoặc cả ba kênh NumPy."""
    red = np.asarray(red, dtype=np.int16)
    green = np.asarray(green, dtype=np.int16)
    blue = np.asarray(blue, dtype=np.int16)

    brightness = (red + green + blue) // 3
    warmth = red - blue
    red_green_gap = red - green
    looks_like_skin = (
        (brightness >= 35) & (brightness <= 240)
        & (warmth >= 8)
        & (red_green_gap >= -10) & (red_green_gap <= 90)
    )
    result = np.where(looks_like_skin, MASK_ON, MASK_OFF).astype(np.uint8)
    return int(result) if result.ndim == 0 else result

In [ ]:
magic_mirror.preview_skin_evidence()

## Vì sao phải nhìn thêm tám pixel xung quanh?

Pixel đỏ giữa ảnh không đạt luật RGB, nên `raw_mask` của nó là `0`. Tám pixel xung quanh đạt luật và có giá trị `255`.
Trước khi đếm, chương trình đổi `255` thành `1`; số `0` vẫn là `0`:

```text
count = 1 + 1 + 1 + 1 + 0 + 1 + 1 + 1 + 1 = 8
8 >= 5  →  skin_mask của pixel giữa = 255
```

Như vậy, pixel giữa vẫn nằm trong vùng được chọn vì có 8 trong 9 pixel đạt luật, nhiều hơn mức cần là 5.
Ta đếm `0/1` vì kết quả dễ đọc hơn việc tính trung bình của `0/255`.

In [ ]:
magic_mirror.show_mechanism("neighbours")

## SciPy đếm hoặc trộn các pixel xung quanh ra sao?

`convolve_layer` nhìn một vùng 3×3 quanh pixel đang tính. Nó nhân từng số trong vùng với số ở cùng vị trí trong
`kernel`, cộng chín kết quả, rồi chia cho `divisor`.

Ví dụ, tám ô xung quanh bằng `10`, ô giữa bằng `90`, còn chín số trong `kernel` đều bằng `1`:

```text
total = 8 × 10 + 90 = 170
new_value = 170 / 9 = 18.89
```

OUTPUT của pixel giữa là `18.89`. Số `90` đã được trộn với tám số `10`, nên pixel giữa bớt sáng.

`ndimage.convolve` làm phép nhân rồi cộng này ở mọi pixel. `mode="nearest"` chỉ nói cách xử lý mép ảnh: nếu vùng 3×3
đi ra ngoài ảnh, SciPy dùng lại giá trị của pixel biên gần nhất.

In [ ]:
magic_mirror.show_convolution_math()

### Nhiệm vụ 1 — Hoàn thành `convolve_layer`

- **Dữ liệu đã cho sẵn:** `layer`, `kernel`, `divisor`; NumPy và SciPy đã được mở.
- **INPUT thật:** không có. Bộ tự chấm sẽ đưa một bảng 5×5 vào hàm.
- **Em cần viết:** đưa `values` và `weights` vào `ndimage.convolve`, rồi chia kết quả cho `divisor`.
- **OUTPUT để kiểm tra:** khi chỉ ô giữa của bảng bằng `9`, chín trọng số bằng `1` và `divisor = 9`, số ở giữa
  bảng kết quả phải bằng `1`. Bảng INPUT vẫn phải giữ số `9` ban đầu.

In [ ]:
def convolve_layer(layer, kernel, divisor):
    """Dùng bảng trọng số với SciPy và trả về một bảng số NumPy mới."""
    values = np.asarray(layer, dtype=np.float32)
    weights = np.asarray(kernel, dtype=np.float32)
    filtered = ndimage.convolve(values, weights, mode="nearest")
    return filtered / divisor

In [ ]:
magic_mirror.preview_library_convolution()

### Nhiệm vụ 3 — Hoàn thành `detect_skin`

- **Dữ liệu đã cho sẵn:** ảnh PIL `img`, bảng 3×3 toàn số `1` và mức tối thiểu `5`.
- **INPUT thật:** một ảnh; ở cuối bài, mỗi khung hình camera là một INPUT mới.
- **Em cần viết:** đổi `raw_mask` từ `0/255` thành `binary` chỉ có `0/1`; dùng `convolve_layer` để đếm; so
  `neighbour_count` với `SKIN_NEIGHBOURS_NEEDED`.
- **OUTPUT để kiểm tra:** một bảng `uint8` chỉ có `0/255`. Pixel đỏ giữa vùng da phải là `255`; pixel giữa ảnh nền
  xanh phải là `0`.

In [ ]:
def detect_skin(img):
    """Tạo ảnh đánh dấu vùng da bằng cách đếm kết quả trong vùng 3x3."""
    pixels = np.asarray(img.convert("RGB"), dtype=np.int16)
    raw_mask = skin_evidence(
        pixels[:, :, 0],
        pixels[:, :, 1],
        pixels[:, :, 2],
    )
    binary = (raw_mask == MASK_ON).astype(np.float32)
    neighbour_count = convolve_layer(binary, SKIN_VOTE_KERNEL, 1)
    return np.where(
        neighbour_count >= SKIN_NEIGHBOURS_NEEDED, MASK_ON, MASK_OFF
    ).astype(np.uint8)

In [ ]:
magic_mirror.preview_skin_mask()

## Tìm một pixel đỏ hơn vùng ngay quanh nó

Màu đỏ mạnh chưa đủ để kết luận, vì cả vùng có thể đang ở dưới ánh sáng đỏ. Ta so pixel giữa với vùng 5×5 quanh nó.
Đầu tiên, chương trình tính một số tên là `redness`.

Với pixel đỏ `(225, 62, 66)`:

```text
redness_spot = 225 - (62 + 66) / 2 = 225 - 64 = 161
```

Với pixel da `(183, 127, 103)`:

```text
redness_skin = 183 - (127 + 103) / 2 = 183 - 115 = 68
```

Trong vùng 5×5 có pixel đỏ ở giữa và 24 pixel da. Trung bình của 25 số `redness` là:

```text
local_redness = (161 + 24 × 68) / 25 = 1793 / 25 = 71.72
red_gap = 161 - 71.72 = 89.28
89.28 >= 24  →  chọn pixel giữa
```

Sau đó, `maximum_filter` mở một ô được chọn thành vùng 3×3. Nhờ vậy, bước làm mềm không chỉ đổi đúng một chấm nhỏ.

In [ ]:
magic_mirror.show_mechanism("red_spot")

### Nhiệm vụ 4 — Hoàn thành `detect_pimples`

- **Dữ liệu đã cho sẵn:** ảnh `img`, `skin_mask`, vùng 5×5 và mốc so sánh `24`.
- **INPUT thật:** ảnh RGB cùng `skin_mask` của ảnh đó.
- **Em cần viết:** đưa `redness` vào `uniform_filter`; đưa `candidate` vào `maximum_filter`.
- **OUTPUT để kiểm tra:** `pimple_mask` là bảng `uint8`; pixel giữa vùng đỏ bằng `255`, còn góc ảnh bằng `0`.

In [ ]:
def detect_pimples(img, skin_mask):
    """Tìm điểm đỏ nổi bật trong vùng 5x5 rồi mở rộng vùng được chọn."""
    pixels = np.asarray(img.convert("RGB"), dtype=np.float32)
    red, green, blue = pixels[:, :, 0], pixels[:, :, 1], pixels[:, :, 2]
    redness = np.maximum(0, red - (green + blue) / 2)
    local_redness = ndimage.uniform_filter(redness, size=5, mode="nearest")
    candidate = (
        (np.asarray(skin_mask) == MASK_ON)
        & (redness - local_redness >= PIMPLE_RED_GAP)
    )
    expanded = ndimage.maximum_filter(candidate, size=3, mode="nearest")
    return np.where(expanded, MASK_ON, MASK_OFF).astype(np.uint8)

In [ ]:
magic_mirror.preview_pimple_mask()

## Tính màu mới, rồi quyết định có dùng màu đó không

Đầu tiên, chương trình tính một màu mềm hơn cho pixel giữa. Bảng 3×3 cho pixel giữa trọng số `4`, bốn pixel
sát cạnh trọng số `2`, bốn pixel ở góc trọng số `1`. Tổng chín trọng số là `16`:

```text
1  2  1
2  4  2      tổng trọng số = 16
1  2  1
```

Pixel giữa là `(225, 62, 66)`. Tám pixel xung quanh đều là `(183, 127, 103)`. Tính riêng từng kênh:

```text
new_red   = (4 × 225 + 12 × 183) / 16 = 3096 / 16 = 193.5 → 194
new_green = (4 ×  62 + 12 × 127) / 16 = 1772 / 16 = 110.75 → 111
new_blue  = (4 ×  66 + 12 × 103) / 16 = 1500 / 16 = 93.75 → 94
```

Màu mềm là `(194, 111, 94)`. Sau đó `np.where` nhìn `pimple_mask` tại đúng vị trí này:

- mask bằng `255` → dùng màu mềm `(194, 111, 94)`;
- mask bằng `0` → giữ màu ban đầu `(225, 62, 66)`.

Vì vậy chương trình có thể tính màu mềm cho cả ảnh, nhưng chỉ thay các pixel đã được chọn.

In [ ]:
magic_mirror.show_mechanism("soften")

### Nhiệm vụ 5 — Hoàn thành `remove_pimples`

- **Dữ liệu đã cho sẵn:** ảnh `img`, bảng trọng số và bốn hàm em vừa hoàn thành.
- **INPUT thật:** một ảnh PIL; cuối bài, đây là khung hình camera vừa nhận được.
- **Em cần viết:** đưa `pixels` vào `ndimage.convolve`; đưa `pimple_mask` vào điều kiện của `np.where`.
  `pimple_mask[:, :, None]` dùng cùng một lựa chọn cho cả ba số R, G, B của mỗi pixel.
- **OUTPUT để kiểm tra:** một ảnh PIL cùng kích thước. Pixel đỏ giữa ảnh bớt nổi bật, pixel ở góc giữ nguyên và ảnh INPUT
  không bị sửa trực tiếp.

In [ ]:
def remove_pimples(img):
    """Làm mềm nơi pimple_mask bằng 255 và giữ nguyên mọi pixel còn lại."""
    source = img.convert("RGB")
    pixels = np.asarray(source, dtype=np.float32)
    skin_mask = detect_skin(source)
    pimple_mask = detect_pimples(source, skin_mask)

    weights = np.asarray(SOFTEN_KERNEL, dtype=np.float32)[:, :, None]
    softened = ndimage.convolve(pixels, weights, mode="nearest") / weights.sum()
    combined = np.where(pimple_mask[:, :, None] == MASK_ON, softened, pixels)
    output = np.clip(np.rint(combined), 0, 255).astype(np.uint8)
    return Image.fromarray(output, "RGB")

In [ ]:
magic_mirror.preview_cleanup()

## Kiểm tra năm hàm

Chạy ô dưới. Mỗi dòng sẽ ghi tên hàm, hàm đã đúng hay chưa và lý do nếu chưa đúng. OUTPUT cuối cùng cần là
`Kết quả: 5/5 phần đã đúng.` Trang tự lưu code, kết quả này và sáu bảng cơ chế để em có thể làm tiếp vào lần sau.

In [ ]:
magic_mirror.check_skin_code()

## Nối năm hàm thành một chương trình

Chạy ô dưới. OUTPUT có sáu hình theo đúng thứ tự dữ liệu đi qua chương trình: ảnh RGB → `skin_mask` → vị trí
`skin_mask` phủ lên ảnh → `pimple_mask` → vị trí `pimple_mask` phủ lên ảnh → ảnh cuối.

Lớp màu phủ cho biết **đúng vị trí pixel** mà mask đã chọn. Nếu vị trí chọn sai, kiểm tra ba phép tính RGB, số đếm 3×3
hoặc độ chênh màu đỏ. Nếu vị trí đúng nhưng màu cuối sai, kiểm tra bảng trọng số và `np.where`.

In [ ]:
magic_mirror.skin_demo()

## Thử thêm vài cách đổi ảnh quen thuộc

Chạy hai ô dưới để xem sáu phép đổi ảnh. Dưới mỗi hình đều có tên phép đổi. Khi so với ảnh gốc, em hãy trả lời:

1. Phép nào chỉ đổi các số của chính pixel đang xét?
2. Phép nào phải lấy thêm số từ các pixel xung quanh?
3. Với bảng làm nét, pixel giữa được nhân với số nào?

In [ ]:
magic_mirror.numpy_filter_gallery()

In [ ]:
magic_mirror.numpy_kernel_gallery()

### Tự sửa một phép đổi màu bằng NumPy

Đoạn code mẫu tăng kênh xanh dương thêm `40` và dùng `np.clip` để giữ số trong khoảng `0..255`.
Giá trị cho sẵn là ảnh mẫu. Dữ liệu bên ngoài (INPUT): không có. Việc cần làm (PROCESS): sao chép bảng số,
đổi đúng một kênh, giới hạn kết quả trong `0..255` rồi trả về. Kết quả đúng (OUTPUT) phải có ảnh trước, ảnh sau,
kích thước và kiểu số của bảng kết quả; ảnh đầu vào không bị sửa. Hãy thử đổi kênh hoặc đổi số cộng thêm.

In [ ]:
def my_numpy_filter(pixels):
    result = pixels.copy().astype(np.int16)
    result[:, :, 2] = np.clip(result[:, :, 2] + 40, 0, 255)
    return result.astype(np.uint8)

magic_mirror.preview_numpy_filter(my_numpy_filter)

## Kiểm chứng bằng ảnh công khai

Ba ảnh CC0 bên dưới đã được lưu sẵn trong bài, nên trang không cần tải ảnh từ nơi khác: hai chân dung có màu da và ánh sáng
khác nhau, cùng một ảnh cận cảnh bề mặt da. Nguồn: [William Stitt](https://commons.wikimedia.org/wiki/File:Face_portrait_(Unsplash).jpg),
[Eddie Kopp](https://commons.wikimedia.org/wiki/File:Young_woman%27s_face_(Unsplash).jpg) và
[Montavius Howard](https://commons.wikimedia.org/wiki/File:Human_skin_close-up.jpg).

Chạy `try_public_photo(0)`, rồi đổi số cuối thành `1` hoặc `2`. Mỗi lần chạy, OUTPUT cho thấy bốn hình có nhãn:
ảnh INPUT, `skin_mask`, `pimple_mask` và ảnh cuối. Hãy nhìn vị trí ô trắng trong hai mask để tìm chỗ luật màu nhận sai.
Mục tiêu là kiểm tra giới hạn của code, không phải nhận xét về người trong ảnh.

In [ ]:
magic_mirror.show_public_photo_gallery()

In [ ]:
magic_mirror.try_public_photo(0)

## Cơ chế 6 — Chỉ đổi pixel khi hai điều kiện cùng đúng

Luật RGB có thể chọn nhầm một vật có màu gần giống da. Face Mesh bổ sung một câu hỏi: pixel này có nằm trong khuôn mặt không?

- `face_mask = 1`: pixel nằm trong đường bao khuôn mặt.
- `skin_mask = 1`: pixel đạt luật màu và đủ số pixel lân cận.

Chương trình tính `allowed = face_mask & skin_mask`. Chỉ trường hợp `1 & 1` cho kết quả `1`. Ba trường hợp còn lại
đều giữ màu ban đầu. Chạy bảng dưới và thử đủ bốn cặp số.

In [ ]:
magic_mirror.show_mechanism("face_gate")

## MediaPipe tạo `face_mask` như thế nào?

Face Mesh nhận ảnh và trả về tối đa 478 điểm trên khuôn mặt. Mỗi điểm có vị trí ngang và dọc. Ta chọn các điểm chạy
quanh viền mặt, trong đó có điểm `10` gần trán, `454` bên phải, `152` gần cằm và `234` bên trái. Nối các điểm này
thành một đường khép kín rồi tô phần bên trong bằng `1`; phần bên ngoài là `0`. Đó là `face_mask`.

```text
face_mask = pixel nằm trong đường bao khuôn mặt
skin_mask = pixel đạt điều kiện RGB và đủ số pixel lân cận
allowed   = face_mask & skin_mask
output    = np.where(allowed[..., None], cleaned, original)
```

`allowed[..., None]` dùng cùng giá trị `allowed` cho cả ba số R, G, B của pixel. Face Mesh chỉ cho biết vùng khuôn mặt;
nó không chẩn đoán da và không tự tìm vùng đỏ.

In [ ]:
magic_mirror.show_face_mesh_map()

In [ ]:
magic_mirror.show_face_mask_pipeline()

## Chạy với INPUT thật từ camera

Mỗi khung hình camera đi qua đúng các bước em vừa học. Trên màn hình, bật **Hiện đường viền Face Mesh** để thấy vùng
MediaPipe tìm được. Chọn **Nét (320×240)** nếu hình còn rỗ và máy chạy ổn; chọn **Cân bằng (240×180)** hoặc
**Tiết kiệm (160×120)** nếu máy xử lý chậm. Trình duyệt phóng kết quả lên 480×360 và làm mượt khi hiển thị, nên độ nét
phụ thuộc cả kích thước xử lý lẫn camera thật.

Ảnh được xử lý ngay trong trình duyệt và không được đưa vào localStorage. Nếu camera bị chặn, em vẫn có thể hoàn thành
toàn bộ bài bằng ảnh 7×7, ảnh tổng hợp và ba ảnh công khai ở trên.

In [ ]:
magic_mirror.run()

## Ghi lại điều em quan sát được

Sau khi thử ảnh mẫu hoặc camera, hãy thêm một ô code hoặc ô chữ và ghi ba ý:

1. Một trường hợp luật nhận đúng vùng cần xử lý.
2. Một vật hoặc ánh sáng làm luật nhận nhầm.
3. Một thay đổi ở bảng trọng số hoặc mốc so sánh và kết quả em nhìn thấy.